# 02 整合与注释

基于 `01_qc_preprocess.ipynb` 生成的 QC-clean `h5ad` 对象，完成：
- 合并全部样本并重建未整合嵌入
- 在可用时比较 `Harmony` 效果
- 基于 marker 参考完成大类细胞注释
- 对上皮细胞进行第一版恶性初筛
- 导出 `malignant_epithelial_screen`、`CAF`、`myeloid`、`T_NK`、`Schwann_neural` 五条后续分析主线

> 当前版本以“analysis-ready object”为目标，不追求最终论文级终版命名。所有 PNI 趋势解释保持探索性表述。

## 1. 参数、路径与依赖检查

运行前请确认：
1. `01_qc_preprocess.ipynb` 已运行完成，并产生 `results/intermediate/qc_clean/*.h5ad`
2. `results/tables/qc/qc_clean_object_index.tsv` 存在且与对象文件一致
3. `references/marker_reference.tsv` 可正常读取

本 notebook 的约定：
- Markdown 说明使用中文
- 代码中的 warning / exception / runtime message 使用英文
- `Harmony` 为可选步骤，缺包时自动跳过而不阻断主流程

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import harmonypy  # noqa: F401

sns.set_context("talk")
sns.set_style("whitegrid")
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=110, facecolor="white")

_CWD = Path.cwd().resolve()
if (_CWD / "notebooks").exists() and (_CWD / "results").exists():
    PROJECT_ROOT = _CWD
elif _CWD.name == "notebooks" and (_CWD.parent / "results").exists():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD.parent
QC_OBJECT_INDEX_PATH = PROJECT_ROOT / "results" / "tables" / "qc" / "qc_clean_object_index.tsv"
MARKER_REFERENCE_PATH = PROJECT_ROOT / "references" / "marker_reference.tsv"
OUTPUT_ROOT = PROJECT_ROOT / "results"
INTERMEDIATE_DIR = OUTPUT_ROOT / "intermediate" / "integration"
TABLE_DIR = OUTPUT_ROOT / "tables" / "integration"
FIGURE_DIR = OUTPUT_ROOT / "figures" / "integration"
RANDOM_SEED = 20260309
N_TOP_GENES = 3000
N_PCS = 30
N_NEIGHBORS = 15
LEIDEN_RESOLUTION_MAIN = 0.6
LEIDEN_RESOLUTION_SUBSET = 0.4
MIN_GENES_FOR_SCORE = 2
SCORING_GENE_POOL_SIZE = 5000

for path in [INTERMEDIATE_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def log_step(title: str) -> None:
    print(f"\n[{title}]")


def print_parameter_summary() -> None:
    summary = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "QC_OBJECT_INDEX_PATH": str(QC_OBJECT_INDEX_PATH),
        "MARKER_REFERENCE_PATH": str(MARKER_REFERENCE_PATH),
        "INTERMEDIATE_DIR": str(INTERMEDIATE_DIR),
        "TABLE_DIR": str(TABLE_DIR),
        "FIGURE_DIR": str(FIGURE_DIR),
        "RANDOM_SEED": RANDOM_SEED,
        "N_TOP_GENES": N_TOP_GENES,
        "N_PCS": N_PCS,
        "N_NEIGHBORS": N_NEIGHBORS,
        "LEIDEN_RESOLUTION_MAIN": LEIDEN_RESOLUTION_MAIN,
        "LEIDEN_RESOLUTION_SUBSET": LEIDEN_RESOLUTION_SUBSET,
    }
    print("Current parameter configuration:")
    for key, value in summary.items():
        print(f"- {key}: {value}")


log_step("Step 1A: print parameter summary")
print_parameter_summary()


In [ ]:
REQUIRED_OBJECT_INDEX_COLUMNS = [
    "sample_id",
    "patient_id",
    "tissue_type",
    "pni_level",
    "object_path",
    "n_cells",
]

REQUIRED_OBS_COLUMNS = [
    "sample_id",
    "patient_id",
    "tissue_type",
    "pni_level",
    "n_genes_by_counts",
    "total_counts",
    "percent_mt",
]

MAJOR_CLASS_ORDER = [
    "epithelial",
    "fibroblast_CAF",
    "myeloid",
    "T_NK",
    "B_plasma",
    "endothelial_pericyte",
    "Schwann_neural",
]

SUBSET_CONFIG = {
    "malignant_epithelial_screen": {
        "mask_major": "epithelial",
        "candidate_labels": [
            "malignant_like_program_high",
            "biliary_like_normal_like",
            "undetermined",
        ],
    },
    "CAF": {
        "mask_major": "fibroblast_CAF",
        "candidate_labels": ["myCAF_mCAF", "iCAF_inflammatory", "ecm_remodeling_CAF"],
    },
    "myeloid": {
        "mask_major": "myeloid",
        "candidate_labels": ["monocyte", "macrophage_TAM", "inflammatory_TAM_SPP1", "neutrophil_like"],
    },
    "T_NK": {
        "mask_major": "T_NK",
        "candidate_labels": ["pan_T", "CD8_cytotoxic", "CD8_exhausted", "Treg", "NK_like"],
    },
    "Schwann_neural": {
        "mask_major": "Schwann_neural",
        "candidate_labels": ["Schwann_core", "glial_like", "neural_interaction_program_high", "undetermined_neural_like"],
    },
}

SPECIAL_GENESETS = {
    "NK_like": ["NKG7", "KLRD1", "GNLY", "TRAC", "CTSW"],
    "Schwann_core": ["SOX10", "S100B", "MPZ", "PLP1", "PMP22", "NGFR"],
    "glial_like": ["GFAP", "PLP1", "S100B", "ERBB3", "SOX10"],
    "neural_interaction_program_high": ["MPZ", "MPZL1", "DAG1", "LAMB3", "ITGB4", "APP", "SORL1"],
    "malignant_like_program_high": ["EPCAM", "KRT8", "KRT18", "KRT19", "MSLN", "MUC1", "CEACAM6", "VIM", "LAMC2"],
    "biliary_like_normal_like": ["KRT19", "EPCAM", "MUC1", "KRT7", "SOX9"],
    "undetermined": ["EPCAM", "KRT19"],
    "undetermined_neural_like": ["S100B", "PLP1"],
}


def validate_columns(df: pd.DataFrame, required_columns: list[str], label: str) -> None:
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} missing required columns: {missing}")


def load_object_index(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"QC object index file not found: {path}")
    object_index = pd.read_csv(path, sep="\t")
    validate_columns(object_index, REQUIRED_OBJECT_INDEX_COLUMNS, "QC object index")
    object_index["sample_id"] = object_index["sample_id"].astype(str)
    object_index = object_index.sort_values("sample_id").reset_index(drop=True)
    return object_index


def load_marker_reference(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"marker reference file not found: {path}")
    marker_reference = pd.read_csv(path, sep="\t")
    required = ["category", "cell_class", "subtype_or_program", "marker_genes", "priority"]
    validate_columns(marker_reference, required, "marker reference")
    marker_reference = marker_reference.copy()
    marker_reference["marker_gene_list"] = marker_reference["marker_genes"].fillna("").apply(
        lambda x: [gene.strip() for gene in str(x).split(",") if gene.strip()]
    )
    return marker_reference


def make_unique_var_names(adata: sc.AnnData) -> sc.AnnData:
    adata = adata.copy()
    adata.var_names_make_unique()
    return adata


def read_clean_objects(object_index: pd.DataFrame) -> dict[str, sc.AnnData]:
    adatas: dict[str, sc.AnnData] = {}
    for row in object_index.to_dict(orient="records"):
        sample_id = row["sample_id"]
        object_path = Path(row["object_path"])
        if not object_path.exists():
            raise FileNotFoundError(f"clean object file not found: {object_path}")
        adata = sc.read_h5ad(object_path)
        adata = make_unique_var_names(adata)
        validate_columns(adata.obs, REQUIRED_OBS_COLUMNS, f"obs for {sample_id}")
        adatas[sample_id] = adata
    return adatas


def merge_clean_objects(adatas: dict[str, sc.AnnData], ordered_samples: list[str]) -> sc.AnnData:
    merged = sc.concat(
        [adatas[sample_id] for sample_id in ordered_samples],
        join="outer",
        label="source_sample",
        keys=ordered_samples,
        index_unique=None,
        merge="same",
    )
    merged.var_names_make_unique()
    return merged


log_step("Step 1B: load object index and marker reference")
object_index = load_object_index(QC_OBJECT_INDEX_PATH)
marker_reference = load_marker_reference(MARKER_REFERENCE_PATH)
clean_adatas = read_clean_objects(object_index)
print(object_index)
print(f"Loaded marker reference rows: {marker_reference.shape[0]}")


## 2. 合并对象并重建未整合嵌入

本部分使用 `01` 的 QC-clean 对象直接合并，不回到原始矩阵。目标是建立：
- 可复跑的未整合 baseline
- 大类注释与恶性初筛所需的程序分数
- 后续 subset 导出的统一主对象

In [ ]:
def preprocess_for_embedding(
    adata: sc.AnnData,
    cluster_key: str,
    n_top_genes: int = N_TOP_GENES,
    n_pcs: int = N_PCS,
    n_neighbors: int = N_NEIGHBORS,
    leiden_resolution: float = LEIDEN_RESOLUTION_MAIN,
) -> sc.AnnData:
    adata = adata.copy()
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=min(n_top_genes, adata.n_vars),
        flavor="seurat",
        subset=False,
    )
    if int(adata.var["highly_variable"].sum()) == 0:
        raise ValueError("no highly variable genes were selected")
    n_comps = min(n_pcs, max(2, adata.n_obs - 1), max(1, int(adata.var["highly_variable"].sum()) - 1))
    sc.tl.pca(
        adata,
        n_comps=n_comps,
        use_highly_variable=True,
        svd_solver="arpack",
        zero_center=False,
        random_state=RANDOM_SEED,
    )
    usable_pcs = min(n_pcs, adata.obsm["X_pca"].shape[1])
    sc.pp.neighbors(adata, n_neighbors=min(n_neighbors, max(2, adata.n_obs - 1)), n_pcs=usable_pcs)
    sc.tl.umap(adata, random_state=RANDOM_SEED)
    assign_cluster_labels(adata, cluster_key=cluster_key, leiden_resolution=leiden_resolution)
    return adata


def assign_cluster_labels(adata: sc.AnnData, cluster_key: str, leiden_resolution: float) -> None:
    sc.tl.leiden(adata, resolution=leiden_resolution, key_added=cluster_key, random_state=RANDOM_SEED)
    adata.uns[f"{cluster_key}_method"] = "leiden"


def create_marker_gene_dict(marker_df: pd.DataFrame, category: str | None = None) -> dict[str, list[str]]:
    filtered = marker_df.copy()
    if category is not None:
        filtered = filtered.loc[filtered["category"] == category].copy()
    gene_dict: dict[str, list[str]] = {}
    for row in filtered.to_dict(orient="records"):
        gene_dict[row["subtype_or_program"]] = row["marker_gene_list"]
    gene_dict.update(SPECIAL_GENESETS)
    return gene_dict


def present_genes(adata: sc.AnnData, genes: list[str]) -> list[str]:
    return [gene for gene in genes if gene in adata.var_names]


def safe_score_genes(adata: sc.AnnData, score_name: str, genes: list[str], ctrl_size: int = 25) -> None:
    genes_present = present_genes(adata, genes)
    if len(genes_present) < MIN_GENES_FOR_SCORE:
        adata.obs[score_name] = 0.0
        return
    sc.tl.score_genes(
        adata,
        gene_list=genes_present,
        score_name=score_name,
        ctrl_size=min(ctrl_size, max(1, len(genes_present))),
        random_state=RANDOM_SEED,
        use_raw=False,
    )


def add_program_scores(adata: sc.AnnData, marker_df: pd.DataFrame) -> list[str]:
    gene_dict = create_marker_gene_dict(marker_df)
    score_columns = []
    for label, genes in gene_dict.items():
        score_name = f"score__{label}"
        safe_score_genes(adata, score_name, genes)
        score_columns.append(score_name)
    return score_columns


log_step("Step 2A: merge QC-clean objects")
ordered_samples = object_index["sample_id"].tolist()
merged = merge_clean_objects(clean_adatas, ordered_samples)
expected_total_cells = int(object_index["n_cells"].sum())
assert merged.n_obs == expected_total_cells, "merged cell count does not match object index"
print(f"Merged object: {merged.n_obs} cells x {merged.n_vars} genes")

log_step("Step 2B: run unintegrated embedding pipeline")
adata_main = preprocess_for_embedding(merged, cluster_key="cluster_unintegrated")
score_columns_main = add_program_scores(adata_main, marker_reference)
adata_main.write_h5ad(INTERMEDIATE_DIR / "integrated_main_unintegrated.h5ad")
print(f"Added program scores: {len(score_columns_main)} columns")



In [ ]:
def save_umap_grid(adata: sc.AnnData, colors: list[str], path: Path, title: str | None = None) -> None:
    sc.pl.umap(adata, color=colors, ncols=2, show=False, frameon=False, save=False)
    fig = plt.gcf()
    if title:
        fig.suptitle(title)
        fig.subplots_adjust(top=0.92)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


log_step("Step 2C: export baseline UMAP figures")
baseline_colors = [
    "cluster_unintegrated",
    "sample_id",
    "patient_id",
    "tissue_type",
    "pni_level",
    "n_genes_by_counts",
    "total_counts",
    "percent_mt",
]
save_umap_grid(
    adata_main,
    baseline_colors,
    FIGURE_DIR / "main_umap_unintegrated_overview.png",
    title="Unintegrated baseline overview",
)
print("Exported: main_umap_unintegrated_overview.png")


## 3. 可选 `Harmony` 比较

`Harmony` 不是本 notebook 的硬前提。若环境中已安装 `harmonypy`，则运行并与未整合 UMAP 对比；若未安装，则明确跳过并继续后续注释。

In [ ]:
def run_harmony_required(adata: sc.AnnData, batch_key: str = "sample_id", cluster_key: str = "cluster_harmony") -> tuple[sc.AnnData, str]:
    adata_harmony = adata.copy()
    sc.external.pp.harmony_integrate(adata_harmony, key=batch_key, basis="X_pca")
    sc.pp.neighbors(
        adata_harmony,
        use_rep="X_pca_harmony",
        n_neighbors=min(N_NEIGHBORS, max(2, adata_harmony.n_obs - 1)),
    )
    sc.tl.umap(adata_harmony, random_state=RANDOM_SEED)
    assign_cluster_labels(adata_harmony, cluster_key=cluster_key, leiden_resolution=LEIDEN_RESOLUTION_MAIN)
    return adata_harmony, "Harmony integration completed successfully."


log_step("Step 3: run Harmony integration")
adata_harmony, harmony_message = run_harmony_required(adata_main, batch_key="sample_id")
print(harmony_message)
adata_harmony.write_h5ad(INTERMEDIATE_DIR / "integrated_main_harmony.h5ad")
save_umap_grid(
        adata_harmony,
        ["cluster_harmony", "sample_id", "patient_id", "tissue_type", "pni_level"],
        FIGURE_DIR / "main_umap_harmony_overview.png",
        title="Harmony overview",
    )
print("Exported: main_umap_harmony_overview.png")


## 4. 大类注释与 cluster marker 汇总

第一版大类注释采用“marker 分数 + cluster 层级摘要”的组合逻辑：
- 先对细胞层面打分
- 再汇总到 cluster
- 用最高支持证据给出 `cell_class_major`

当前输出偏向可复跑和可交接，不取代后续人工复核。

In [ ]:
def rank_markers(adata: sc.AnnData, groupby: str) -> pd.DataFrame:
    sc.tl.rank_genes_groups(adata, groupby=groupby, method="wilcoxon", use_raw=False)
    marker_df = sc.get.rank_genes_groups_df(adata, group=None)
    marker_df = marker_df.rename(columns={"group": groupby})
    return marker_df


def build_major_class_score_map(marker_df: pd.DataFrame) -> dict[str, list[str]]:
    annotation_rows = marker_df.loc[marker_df["category"] == "annotation"].copy()
    major_map: dict[str, list[str]] = {}
    for major_class in MAJOR_CLASS_ORDER:
        rows = annotation_rows.loc[annotation_rows["cell_class"] == major_class]
        score_names = [f"score__{label}" for label in rows["subtype_or_program"].tolist()]
        score_names = [score for score in score_names if score in adata_main.obs.columns]
        major_map[major_class] = score_names
    return major_map


def assign_major_classes(adata: sc.AnnData, marker_df: pd.DataFrame, cluster_key: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    major_map = build_major_class_score_map(marker_df)
    score_summary_records = []
    mapping_records = []
    clusters = adata.obs[cluster_key].astype(str)
    for cluster in sorted(clusters.unique(), key=lambda x: (len(x), x)):
        cluster_mask = clusters == cluster
        cluster_obs = adata.obs.loc[cluster_mask].copy()
        cluster_size = int(cluster_mask.sum())
        major_scores = {}
        for major_class, score_columns in major_map.items():
            if not score_columns:
                major_scores[major_class] = -np.inf
                continue
            major_scores[major_class] = float(cluster_obs[score_columns].mean().mean())
        sorted_scores = sorted(major_scores.items(), key=lambda item: item[1], reverse=True)
        best_label, best_score = sorted_scores[0]
        second_score = sorted_scores[1][1] if len(sorted_scores) > 1 else -np.inf
        confidence = "high" if (best_score - second_score) >= 0.05 else "medium"
        tumor_fraction = float((cluster_obs["tissue_type"] == "tumor").mean())
        adjacent_fraction = float((cluster_obs["tissue_type"] == "adjacent").mean())
        mapping_records.append({
            cluster_key: cluster,
            "cell_class_major": best_label,
            "cell_class_confidence": confidence,
            "annotation_note": f"auto_v1_from_marker_scores; tumor_fraction={tumor_fraction:.3f}; adjacent_fraction={adjacent_fraction:.3f}",
            "n_cells": cluster_size,
            "tumor_fraction": tumor_fraction,
            "adjacent_fraction": adjacent_fraction,
        })
        record = {cluster_key: cluster}
        record.update({f"major_score__{key}": value for key, value in major_scores.items()})
        score_summary_records.append(record)
    mapping_df = pd.DataFrame(mapping_records)
    score_summary_df = pd.DataFrame(score_summary_records)
    return mapping_df, score_summary_df


def apply_cluster_mapping(adata: sc.AnnData, cluster_key: str, mapping_df: pd.DataFrame) -> None:
    mapping = mapping_df.set_index(cluster_key)
    for column in ["cell_class_major", "cell_class_confidence", "annotation_note"]:
        adata.obs[column] = adata.obs[cluster_key].astype(str).map(mapping[column])


log_step("Step 4A: rank markers for main clusters")
main_marker_df = rank_markers(adata_main, groupby="cluster_unintegrated")
main_marker_df.to_csv(TABLE_DIR / "main_cluster_markers_unintegrated.tsv", sep="\t", index=False)

log_step("Step 4B: assign major cell classes")
mapping_df, major_score_summary_df = assign_major_classes(adata_main, marker_reference, cluster_key="cluster_unintegrated")
apply_cluster_mapping(adata_main, "cluster_unintegrated", mapping_df)
adata_main.write_h5ad(INTERMEDIATE_DIR / "integrated_main_annotated.h5ad")
mapping_df.to_csv(TABLE_DIR / "cluster_to_major_cellclass_mapping.tsv", sep="\t", index=False)
major_score_summary_df.to_csv(TABLE_DIR / "cluster_major_score_summary.tsv", sep="\t", index=False)
print(mapping_df)


In [ ]:
log_step("Step 4C: export major-annotation plots")
save_umap_grid(
    adata_main,
    ["cluster_unintegrated", "cell_class_major", "cell_class_confidence", "tissue_type"],
    FIGURE_DIR / "main_umap_major_annotation.png",
    title="Major cell-class annotation",
)

major_marker_rows = marker_reference.loc[
    (marker_reference["category"] == "annotation")
    & (marker_reference["cell_class"].isin(MAJOR_CLASS_ORDER))
].copy()
major_marker_genes = []
for genes in major_marker_rows["marker_gene_list"].tolist():
    major_marker_genes.extend(genes)
major_marker_genes = sorted(set(present_genes(adata_main, major_marker_genes)))
if major_marker_genes:
    sc.pl.dotplot(
        adata_main,
        var_names=major_marker_genes,
        groupby="cell_class_major",
        show=False,
        save=False,
    )
    plt.gcf().savefig(FIGURE_DIR / "main_dotplot_major_markers.png", bbox_inches="tight")
    plt.close(plt.gcf())
print("Exported: main_umap_major_annotation.png and main_dotplot_major_markers.png")


## 5. 上皮细胞恶性初筛（v1）

当前只做启发式初筛，不用 CNV 证据替代：
- `likely_malignant`
- `epithelial_undetermined`
- `likely_normal_epithelial`

判读综合 marker、机制程序分数与 `tumor/adjacent` 富集，不把增殖或 stress 极端簇直接等同恶性。

In [ ]:
def compute_cluster_tissue_composition(adata: sc.AnnData, cluster_key: str) -> pd.DataFrame:
    summary = (
        adata.obs.groupby(cluster_key, observed=True)["tissue_type"]
        .value_counts(normalize=True)
        .rename("fraction")
        .reset_index()
        .pivot(index=cluster_key, columns="tissue_type", values="fraction")
        .fillna(0.0)
        .reset_index()
    )
    for column in ["tumor", "adjacent"]:
        if column not in summary.columns:
            summary[column] = 0.0
    return summary


def add_malignancy_screen_v1(adata: sc.AnnData, cluster_key: str) -> pd.DataFrame:
    adata.obs["epithelial_flag"] = np.where(adata.obs["cell_class_major"] == "epithelial", "epithelial", "non_epithelial")
    adata.obs["malignancy_screen_v1"] = "non_epithelial"
    adata.obs["malignancy_evidence"] = "not_applicable"

    epithelial = adata[adata.obs["cell_class_major"] == "epithelial"].copy()
    if epithelial.n_obs == 0:
        return pd.DataFrame(columns=[cluster_key, "malignancy_screen_v1", "malignancy_evidence"])

    tissue_comp = compute_cluster_tissue_composition(epithelial, cluster_key)
    cluster_records = []
    for cluster in sorted(epithelial.obs[cluster_key].astype(str).unique(), key=lambda x: (len(x), x)):
        cluster_obs = epithelial.obs.loc[epithelial.obs[cluster_key].astype(str) == cluster].copy()
        mean_malignant = float(cluster_obs.get("score__malignant_support", pd.Series(0, index=cluster_obs.index)).mean())
        mean_normal = float(cluster_obs.get("score__normal_biliary_like", pd.Series(0, index=cluster_obs.index)).mean())
        mean_emt = float(cluster_obs.get("score__EMT_invasion", pd.Series(0, index=cluster_obs.index)).mean())
        mean_egfr = float(cluster_obs.get("score__EGFR_AREG_axis", pd.Series(0, index=cluster_obs.index)).mean())
        mean_neural = float(cluster_obs.get("score__axonogenesis_neural", pd.Series(0, index=cluster_obs.index)).mean())
        fractions_row = tissue_comp.loc[tissue_comp[cluster_key].astype(str) == cluster].iloc[0]
        tumor_fraction = float(fractions_row.get("tumor", 0.0))
        adjacent_fraction = float(fractions_row.get("adjacent", 0.0))
        malignant_axis = mean_malignant + mean_emt + mean_egfr + 0.5 * mean_neural
        normal_axis = mean_normal + 0.25 * adjacent_fraction
        if tumor_fraction >= 0.60 and malignant_axis > normal_axis + 0.10:
            label = "likely_malignant"
        elif adjacent_fraction >= 0.60 and mean_normal >= mean_malignant:
            label = "likely_normal_epithelial"
        else:
            label = "epithelial_undetermined"
        evidence = (
            f"tumor_fraction={tumor_fraction:.3f};adjacent_fraction={adjacent_fraction:.3f};"
            f"malignant_support={mean_malignant:.3f};normal_biliary_like={mean_normal:.3f};"
            f"EMT_invasion={mean_emt:.3f};EGFR_AREG_axis={mean_egfr:.3f};axonogenesis_neural={mean_neural:.3f}"
        )
        cluster_records.append({
            cluster_key: cluster,
            "malignancy_screen_v1": label,
            "malignancy_evidence": evidence,
            "tumor_fraction": tumor_fraction,
            "adjacent_fraction": adjacent_fraction,
        })
    cluster_df = pd.DataFrame(cluster_records)
    label_map = cluster_df.set_index(cluster_key)["malignancy_screen_v1"]
    evidence_map = cluster_df.set_index(cluster_key)["malignancy_evidence"]
    mask = adata.obs["cell_class_major"] == "epithelial"
    adata.obs.loc[mask, "malignancy_screen_v1"] = adata.obs.loc[mask, cluster_key].astype(str).map(label_map)
    adata.obs.loc[mask, "malignancy_evidence"] = adata.obs.loc[mask, cluster_key].astype(str).map(evidence_map)
    return cluster_df


log_step("Step 5: run epithelial malignancy screening v1")
malignancy_cluster_summary = add_malignancy_screen_v1(adata_main, cluster_key="cluster_unintegrated")
adata_main.write_h5ad(INTERMEDIATE_DIR / "integrated_main_annotated.h5ad")
malignancy_cluster_summary.to_csv(TABLE_DIR / "epithelial_malignancy_screen_summary.tsv", sep="\t", index=False)
print(malignancy_cluster_summary)


In [ ]:
save_umap_grid(
    adata_main,
    ["cell_class_major", "malignancy_screen_v1", "tissue_type", "pni_level"],
    FIGURE_DIR / "main_umap_malignancy_screen.png",
    title="Epithelial malignancy screening v1",
)
print("Exported: main_umap_malignancy_screen.png")


## 6. 五条 lineage subset 导出

每个 subset 的目标是形成第一版可继续深挖的对象，而不是在本 notebook 中完成全部终版分析。

统一策略：
- 从主对象按 `cell_class_major` 或恶性标签切出子集
- 在 subset 内重跑轻量嵌入与聚类
- 根据对应 marker 程序分数给出 `subset_subtype_v1`
- 导出对象、marker 表和关键图

In [ ]:
def choose_subset_mask(adata: sc.AnnData, subset_name: str) -> np.ndarray:
    if subset_name == "malignant_epithelial_screen":
        return (adata.obs["cell_class_major"] == "epithelial").to_numpy()
    if subset_name == "Schwann_neural":
        major_mask = (adata.obs["cell_class_major"] == "Schwann_neural").to_numpy()
        if int(major_mask.sum()) > 0:
            return major_mask
        neural_genes = [gene for gene in sorted(set(SPECIAL_GENESETS["Schwann_core"] + SPECIAL_GENESETS["glial_like"])) if gene in adata.var_names]
        if neural_genes:
            expr = adata[:, neural_genes].X
            expr = expr.toarray() if hasattr(expr, "toarray") else np.asarray(expr)
            expr_count = (expr > 0).sum(axis=1)
            expr_mask = np.asarray(expr_count).reshape(-1) >= 2
            if int(expr_mask.sum()) > 0:
                return expr_mask
        score_cols = [col for col in ["score__Schwann_core", "score__glial_like", "score__neural_interaction_program_high"] if col in adata.obs.columns]
        if score_cols:
            combined = adata.obs[score_cols].sum(axis=1)
            threshold = float(combined.quantile(0.99))
            score_mask = (combined >= threshold).to_numpy()
            if int(score_mask.sum()) > 0:
                return score_mask
        return np.zeros(adata.n_obs, dtype=bool)
    major = SUBSET_CONFIG[subset_name]["mask_major"]
    return (adata.obs["cell_class_major"] == major).to_numpy()


def subset_gene_programs(subset_name: str, marker_df: pd.DataFrame) -> dict[str, list[str]]:
    programs: dict[str, list[str]] = {}
    if subset_name == "malignant_epithelial_screen":
        for label in ["malignant_like_program_high", "biliary_like_normal_like", "EMT_invasion", "EGFR_AREG_axis", "axonogenesis_neural", "undetermined"]:
            if label in SPECIAL_GENESETS:
                programs[label] = SPECIAL_GENESETS[label]
            else:
                rows = marker_df.loc[marker_df["subtype_or_program"] == label, "marker_gene_list"].tolist()
                if rows:
                    programs[label] = rows[0]
        return programs
    major_class = SUBSET_CONFIG[subset_name]["mask_major"]
    annotation_rows = marker_df.loc[
        (marker_df["category"] == "annotation") & (marker_df["cell_class"] == major_class)
    ].copy()
    for row in annotation_rows.to_dict(orient="records"):
        programs[row["subtype_or_program"]] = row["marker_gene_list"]
    for label in SUBSET_CONFIG[subset_name]["candidate_labels"]:
        if label in SPECIAL_GENESETS:
            programs[label] = SPECIAL_GENESETS[label]
    if subset_name == "Schwann_neural":
        mechanism_rows = marker_df.loc[
            (marker_df["category"] == "mechanism") & (marker_df["cell_class"] == "Schwann_neural")
        ]
        for row in mechanism_rows.to_dict(orient="records"):
            programs[row["subtype_or_program"]] = row["marker_gene_list"]
    if subset_name == "T_NK" and "NK_like" not in programs:
        programs["NK_like"] = SPECIAL_GENESETS["NK_like"]
    return programs


def score_subset_programs(adata: sc.AnnData, programs: dict[str, list[str]]) -> list[str]:
    score_columns = []
    for label, genes in programs.items():
        score_name = f"subset_score__{label}"
        safe_score_genes(adata, score_name, genes)
        score_columns.append(score_name)
    return score_columns


def infer_subset_subtypes(adata: sc.AnnData, subset_name: str, cluster_key: str) -> pd.DataFrame:
    candidate_labels = SUBSET_CONFIG[subset_name]["candidate_labels"]
    records = []
    for cluster in sorted(adata.obs[cluster_key].astype(str).unique(), key=lambda x: (len(x), x)):
        cluster_obs = adata.obs.loc[adata.obs[cluster_key].astype(str) == cluster].copy()
        label_scores = {}
        for label in candidate_labels:
            score_name = f"subset_score__{label}"
            label_scores[label] = float(cluster_obs[score_name].mean()) if score_name in cluster_obs.columns else -np.inf
        ranked = sorted(label_scores.items(), key=lambda item: item[1], reverse=True)
        best_label, best_score = ranked[0]
        second_score = ranked[1][1] if len(ranked) > 1 else -np.inf
        confidence = "high" if (best_score - second_score) >= 0.05 else "medium"
        if subset_name == "malignant_epithelial_screen":
            tumor_fraction = float((cluster_obs["tissue_type"] == "tumor").mean())
            malignant_score = float(cluster_obs.get("subset_score__malignant_like_program_high", pd.Series(0, index=cluster_obs.index)).mean())
            biliary_score = float(cluster_obs.get("subset_score__biliary_like_normal_like", pd.Series(0, index=cluster_obs.index)).mean())
            if tumor_fraction >= 0.60 and malignant_score > biliary_score:
                best_label = "malignant_like_program_high"
            elif tumor_fraction <= 0.40 and biliary_score >= malignant_score:
                best_label = "biliary_like_normal_like"
            else:
                best_label = "undetermined"
        if subset_name == "Schwann_neural":
            schwann_score = float(cluster_obs.get("subset_score__Schwann_core", pd.Series(0, index=cluster_obs.index)).mean())
            glial_score = float(cluster_obs.get("subset_score__glial_like", pd.Series(0, index=cluster_obs.index)).mean())
            interaction_score = float(cluster_obs.get("subset_score__neural_interaction_program_high", pd.Series(0, index=cluster_obs.index)).mean())
            if interaction_score > max(schwann_score, glial_score) + 0.02:
                best_label = "neural_interaction_program_high"
            elif schwann_score >= glial_score:
                best_label = "Schwann_core"
            elif glial_score > schwann_score:
                best_label = "glial_like"
            else:
                best_label = "undetermined_neural_like"
        records.append({
            cluster_key: cluster,
            "subset_subtype_v1": best_label,
            "subset_subtype_confidence": confidence,
            "n_cells": int(cluster_obs.shape[0]),
            **{f"score__{key}": value for key, value in label_scores.items()},
        })
    return pd.DataFrame(records)


def build_subset_object(adata_main: sc.AnnData, subset_name: str, marker_df: pd.DataFrame) -> dict[str, object]:
    mask = choose_subset_mask(adata_main, subset_name)
    subset = adata_main[mask].copy()
    if subset.n_obs == 0:
        raise ValueError(f"subset has zero cells: {subset_name}")
    if "counts" in subset.layers:
        subset.X = subset.layers["counts"].copy()
    subset.obs["subset_name"] = subset_name
    subset = preprocess_for_embedding(
        subset,
        cluster_key="subset_cluster",
        n_top_genes=min(N_TOP_GENES, subset.n_vars),
        n_pcs=min(N_PCS, max(5, subset.n_obs - 1)),
        n_neighbors=min(N_NEIGHBORS, max(2, subset.n_obs - 1)),
        leiden_resolution=LEIDEN_RESOLUTION_SUBSET,
    )
    programs = subset_gene_programs(subset_name, marker_df)
    score_columns = score_subset_programs(subset, programs)
    subtype_df = infer_subset_subtypes(subset, subset_name=subset_name, cluster_key="subset_cluster")
    subtype_map = subtype_df.set_index("subset_cluster")
    subset.obs["subset_subtype_v1"] = subset.obs["subset_cluster"].astype(str).map(subtype_map["subset_subtype_v1"])
    subset.obs["subset_subtype_confidence"] = subset.obs["subset_cluster"].astype(str).map(subtype_map["subset_subtype_confidence"])
    marker_df_subset = rank_markers(subset, groupby="subset_cluster")
    return {
        "adata": subset,
        "score_columns": score_columns,
        "subtype_table": subtype_df,
        "marker_table": marker_df_subset,
    }


log_step("Step 6: build and export five lineage subsets")
subset_results: dict[str, dict[str, object]] = {}
for subset_name in SUBSET_CONFIG:
    print(f"Running subset pipeline: {subset_name}")
    subset_results[subset_name] = build_subset_object(adata_main, subset_name=subset_name, marker_df=marker_reference)
    subset_adata = subset_results[subset_name]["adata"]
    subset_path = INTERMEDIATE_DIR / f"subset__{subset_name}.h5ad"
    subset_adata.write_h5ad(subset_path)
    subset_results[subset_name]["marker_table"].to_csv(
        TABLE_DIR / f"subset_markers__{subset_name}.tsv", sep="\t", index=False
    )
    subset_results[subset_name]["subtype_table"].to_csv(
        TABLE_DIR / f"subset_cluster_annotation__{subset_name}.tsv", sep="\t", index=False
    )
    save_umap_grid(
        subset_adata,
        ["subset_cluster", "subset_subtype_v1", "tissue_type", "pni_level"],
        FIGURE_DIR / f"subset_umap__{subset_name}.png",
        title=f"Subset overview: {subset_name}",
    )
    print(f"Exported subset object: {subset_path.name}")


In [ ]:
log_step("Step 6B: export subset dotplots")
for subset_name, payload in subset_results.items():
    subset_adata = payload["adata"]
    programs = subset_gene_programs(subset_name, marker_reference)
    genes = []
    for gene_list in programs.values():
        genes.extend(gene_list)
    genes = sorted(set(present_genes(subset_adata, genes)))
    if not genes:
        print(f"No marker genes present for subset dotplot: {subset_name}")
        continue
    sc.pl.dotplot(
        subset_adata,
        var_names=genes,
        groupby="subset_subtype_v1",
        show=False,
        save=False,
    )
    plt.gcf().savefig(FIGURE_DIR / f"subset_dotplot__{subset_name}.png", bbox_inches="tight")
    plt.close(plt.gcf())
print("Subset dotplots exported.")


## 7. 结果汇总与 handoff checklist

本部分导出统一 summary，作为后续 `03+` notebook 的直接入口。

In [ ]:
log_step("Step 7A: write summary tables and manifest")
subset_summary_records = []
for subset_name, payload in subset_results.items():
    subset_adata = payload["adata"]
    for subtype, n_cells in subset_adata.obs["subset_subtype_v1"].value_counts().items():
        subset_summary_records.append({
            "subset_name": subset_name,
            "subset_subtype_v1": subtype,
            "n_cells": int(n_cells),
        })
subset_summary_df = pd.DataFrame(subset_summary_records).sort_values(["subset_name", "n_cells"], ascending=[True, False])
subset_summary_df.to_csv(TABLE_DIR / "subset_cellcount_summary.tsv", sep="\t", index=False)

main_obs_export = adata_main.obs.reset_index().rename(columns={"index": "cell_barcode"})
main_obs_export.to_csv(TABLE_DIR / "main_obs_annotation_export.tsv", sep="\t", index=False)

manifest = {
    "date": "2026-03-09",
    "inputs": {
        "qc_object_index": str(QC_OBJECT_INDEX_PATH),
        "marker_reference": str(MARKER_REFERENCE_PATH),
    },
    "outputs": {
        "main_object": str(INTERMEDIATE_DIR / "integrated_main_annotated.h5ad"),
        "subset_objects": {name: str(INTERMEDIATE_DIR / f"subset__{name}.h5ad") for name in SUBSET_CONFIG},
        "table_dir": str(TABLE_DIR),
        "figure_dir": str(FIGURE_DIR),
    },
}
with open(TABLE_DIR / "run_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)
print(subset_summary_df)


In [ ]:
log_step("Step 7B: handoff checklist")
print("Handoff-ready outputs:")
print("- Main integrated/annotated object: results/intermediate/integration/integrated_main_annotated.h5ad")
print("- Main annotation export: results/tables/integration/main_obs_annotation_export.tsv")
print("- Cluster-to-major mapping: results/tables/integration/cluster_to_major_cellclass_mapping.tsv")
print("- Epithelial malignancy screening summary: results/tables/integration/epithelial_malignancy_screen_summary.tsv")
print("- Five subset objects and their marker tables are ready for downstream notebooks.")
print("Recommended next entries:")
print("1. malignant_epithelial_screen -> malignant program refinement / CNV-based upgrade")
print("2. CAF -> mCAF / iCAF / ECM remodeling deep dive")
print("3. myeloid -> TAM state and inflammatory axis deep dive")
print("4. T_NK -> cytotoxic / exhausted / Treg refinement")
print("5. Schwann_neural -> neural interaction program refinement")
